# What does the client's scoring rule do with a policy?

This notebook loads the bookings, adds the two columns the scoring rule needs, and scores the simplest policy there is: sell nothing extra. It runs in under a minute. Change nothing here; your team's work goes in `src/`.

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

import contract
import data

In [2]:
bookings = data.read_raw()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
print(f"{len(bookings):,} bookings, arrivals {bookings.arrival_date.min().date()} to {bookings.arrival_date.max().date()}")

119,390 bookings, arrivals 2015-07-01 to 2017-08-31


In [3]:
train, choose, proof = contract.split_periods(bookings)
for name, part in [("train", train), ("choose", choose), ("proof", proof)]:
    print(f"{name:>6}: {len(part):>7,} bookings")

 train:  49,157 bookings
choose:  29,546 bookings
 proof:  40,687 bookings


## What does selling nothing extra cost on the choose period?

Every cancellation leaves a room empty and nobody is walked. This is a floor, not the client's rule: the client already oversells with a flat rate, and your savings are measured against that.

In [4]:
nights = contract.nightly_table(choose, prob=np.zeros(len(choose)))
nights.head()

book  cancelled  expected  spread  stay_value
hotel      arrival_date                                               
City Hotel 2016-07-01     141         68       0.0     0.0  319.747589
           2016-07-02      80         33       0.0     0.0  397.106875
           2016-07-03      85         21       0.0     0.0  478.769529
           2016-07-04      92         30       0.0     0.0  440.365543
           2016-07-05      74         14       0.0     0.0  405.705811

In [5]:
contract.night_cost(nights, oversell=np.zeros(len(nights)))

{'cost_eur': 4155340.86,
 'empty_rooms': 10894,
 'walked_guests': 0,
 'walks_per_1000': 0.0,
 'ceiling_met': True,
 'nights': 368}

## What comes next?

- Check every number in `BRIEF.md` against `bookings`, and record each one in the charter.
- Formalise the client's flat rule in `src/backtest.py` and score it on these same nights.
- Leave the proof period alone until every setting is frozen.